# Figure 4 — Population-scale EHR validation

## Data availability

All inputs for this notebook are **copied into** `For Reviewer/source_data/` (or shown from `illustrations/` when a panel cannot be recomputed).

- No Zenodo download is required.
- No paths outside `For Reviewer/` are used after packaging.
- See `DATA_AVAILABILITY.md` and `source_data/manifest.csv` for origins and checksums.

**Files used below** are listed in each panel section.

- `TableS3_Mount_Sinai_Drug_Cancer.csv`
- `TableS4_UK_Biobank_Drug_Disease.csv`

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

from linkd_repro import paths, style, io, illustrate
style.apply()
paths.ensure_output_dirs()
print("For Reviewer root:", paths.ROOT)
print("source_data OK:", paths.SOURCE.exists())

## Panel a — Framework schematic

In [ ]:
illustrate.show_panel('fig4_a', title='Panel a')

## Panel b — Drug–disease OR network (top protective / risk edges)

In [ ]:

ms = io.read_ehr_ms()
# Use logit_or and logit_p
ms = ms.dropna(subset=["logit_or", "logit_p"]).copy()
ms["neglog10p"] = -np.log10(ms["logit_p"].clip(lower=1e-300))
# Keep strongest associations
top = pd.concat([
    ms.nsmallest(40, "logit_or"),
    ms.nlargest(40, "logit_or"),
]).drop_duplicates()
# Simple bipartite layout
drugs = top["Drug Name"].astype(str).unique()
diseases = top["Disease Description"].astype(str).unique()
drug_y = {d: i for i, d in enumerate(drugs)}
dis_y = {d: i for i, d in enumerate(diseases)}
fig, ax = plt.subplots(figsize=(7.5, 5))
for _, r in top.iterrows():
    y0 = drug_y[str(r["Drug Name"])]
    y1 = dis_y[str(r["Disease Description"])]
    color = "#C44E52" if r["logit_or"] < 0 else "#4C72B0"
    ax.plot([0, 1], [y0, y1], color=color, alpha=0.25, lw=0.8)
ax.scatter(np.zeros(len(drugs)), range(len(drugs)), s=10, c="k")
ax.scatter(np.ones(len(diseases)), range(len(diseases)), s=10, c="k")
ax.set_xticks([0, 1])
ax.set_xticklabels(["Drug", "Disease"])
ax.set_yticks([])
ax.set_title("Fig 4b — top Mount Sinai drug–disease edges (red=protective logit_or<0)")
fig.tight_layout()
out = style.save_panel(fig, "fig4_b_network", top[["Drug Name", "Disease Description", "logit_or", "logit_p", "ICD10"]])
plt.show()
print(out)
